# CAVAS — Distributed Deep Learning with Horovod
### IDS Intrusion Detection: TabNet (Tabular) + TFT (Time Series)

**Identical pipeline to `cavas_model_analysis(1).ipynb` but with Horovod for distributed training.**

**Dual prediction targets:**
- `label_generic` → Binary classification (benign / malicious)
- `Label` → Multiclass classification (benign / attack type)

**Models:**
- **TabNet** — sequential attention, native feature importance (distributed via Horovod + PyTorch)
- **Temporal Fusion Transformer (TFT)** — variable selection networks + temporal attention (distributed via Horovod + PyTorch Lightning)

**Hyperparameter tuning:** Optuna (TPE sampler) — run on rank 0 only, best params broadcast to all workers

---

### How to launch this notebook distributed

**Option A — Single machine, multiple GPUs:**
```bash
horovodrun -np 4 -H localhost:4 python cavas_model_horovod_script.py
```

**Option B — Multiple machines:**
```bash
horovodrun -np 8 -H server1:4,server2:4 python cavas_model_horovod_script.py
```

**Option C — Inside Jupyter (single-machine, uses `horovod.spark` or sequential with hvd.local_rank):**
Run cells below — Horovod initializes in-process for prototyping.
For true multi-node, export this notebook as `.py` and launch with `horovodrun`.

In [ ]:
LOCAL_RUN = False
RANDOM_SEED = 42

## 0. Configuration & Setup

In [ ]:
!pip install -q pytorch-tabnet pytorch-forecasting pytorch-lightning optuna optuna-integration scikit-learn pandas pyarrow horovod

In [ ]:
import os, subprocess, warnings, json
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

# Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer

# Sklearn
from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import StandardScaler, LabelEncoder
from sklearn.metrics          import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, matthews_corrcoef
)

# TabNet — multi-task classifier
from pytorch_tabnet.multitask import TabNetMultiTaskClassifier

# PyTorch
import torch
import torch.optim as optim

# TFT
import pytorch_lightning as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data    import GroupNormalizer
from pytorch_forecasting.metrics import CrossEntropy, MultiLoss
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# ══════════════════════════════════════════════════════════════════
# HOROVOD
# ══════════════════════════════════════════════════════════════════
import horovod.torch as hvd
import horovod.pytorch_lightning as hvd_pl   # PL strategy for TFT

# Initialize Horovod
hvd.init()

# Pin GPU (if available) — each worker gets one GPU
if torch.cuda.is_available():
    torch.cuda.set_device(hvd.local_rank())
    DEVICE = torch.device(f'cuda:{hvd.local_rank()}')
else:
    DEVICE = torch.device('cpu')

IS_RANK_0 = (hvd.rank() == 0)

# Optuna
import optuna
from optuna.integration import PyTorchLightningPruningCallback
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
pl.seed_everything(RANDOM_SEED)

if IS_RANK_0:
    print(f"All imports OK")
    print(f"Horovod: {hvd.size()} workers | rank={hvd.rank()} | local_rank={hvd.local_rank()}")
    print(f"Device: {DEVICE}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
PATH_IMG = "images"

if LOCAL_RUN:
    PATH = "DATASETS"
    PERCENTAGE_TO_USE = 0.001
    N_OPTUNA_TRIALS   = 10
    TABNET_MAX_EPOCHS = 50
    TFT_MAX_EPOCHS    = 30
else:
    PATH = "file:///home/PuckTrickadmin/DATASETS"
    PERCENTAGE_TO_USE = 1.0
    N_OPTUNA_TRIALS   = 100
    TABNET_MAX_EPOCHS = 200
    TFT_MAX_EPOCHS    = 100

RANDOM_SEED  = 42
TEST_SIZE    = 0.2
VAL_SIZE     = 0.1

if IS_RANK_0:
    os.makedirs(PATH_IMG, exist_ok=True)
    os.makedirs("models",  exist_ok=True)

In [ ]:
# ── Spark session (only rank 0 for data loading) ──────────────────
# In distributed mode, rank 0 loads + preprocesses data,
# then we broadcast the numpy arrays to all workers.

if IS_RANK_0:
    if LOCAL_RUN:
        java_home = os.environ.get('JAVA_HOME', '')
        if not java_home:
            try:
                java_path = subprocess.check_output(['which', 'java'], text=True).strip()
                os.environ['JAVA_HOME'] = os.path.dirname(os.path.dirname(os.path.realpath(java_path)))
            except subprocess.CalledProcessError:
                print("⚠️  Java not found — run: sudo apt install default-jdk")

        os.environ['PYSPARK_PYTHON']        = 'python3'
        os.environ['PYSPARK_DRIVER_PYTHON'] = 'python3'

        spark = SparkSession.builder \
            .appName("CAVAS_Models_Horovod") \
            .master("local[*]") \
            .config("spark.driver.memory",          "24g") \
            .config("spark.driver.host",            "localhost") \
            .config("spark.ui.showConsoleProgress", "false") \
            .getOrCreate()
    else:
        MASTER_URL  = "spark://10.0.1.8:7077"
        DRIVER_HOST = "10.0.1.8"

        spark = SparkSession.builder \
            .appName("CAVAS_Models_Horovod") \
            .master(MASTER_URL) \
            .config("spark.submit.deployMode",      "client") \
            .config("spark.executor.instances",     "4") \
            .config("spark.executor.cores",         "4") \
            .config("spark.executor.memory",        "13g") \
            .config("spark.driver.memory",          "8g") \
            .config("spark.driver.host",            DRIVER_HOST) \
            .config("spark.driver.bindAddress",     DRIVER_HOST) \
            .config("spark.sql.shuffle.partitions", "32") \
            .getOrCreate()
        spark.sparkContext.setLogLevel("WARN")

    print(f"✅  Spark {spark.version} ready (rank 0 only)")

## 1. Data Loading

In [ ]:
# ── Load data on rank 0, broadcast to all workers ─────────────────
if IS_RANK_0:
    sdf_full = spark.read.parquet(f'{PATH}/all_elaborated.parquet')

    if PERCENTAGE_TO_USE < 1.0:
        fractions = {
            row['label_generic']: PERCENTAGE_TO_USE
            for row in sdf_full.select('label_generic').distinct().collect()
        }
        sdf = sdf_full.sampleBy('label_generic', fractions=fractions, seed=RANDOM_SEED)
        print(f"📦  Stratified {PERCENTAGE_TO_USE*100:.0f}% sample → {sdf.count():,} rows")
    else:
        sdf = sdf_full
        print(f"📦  Full dataset → {sdf.count():,} rows")

    print(f"🧮  Columns: {len(sdf.columns)}")

## 2. Preprocessing — Normalization & Encoding

In [ ]:
# ── Feature metadata (identical to original notebook) ──────────────
FEATURES_LABEL_GENERIC = [
    'Tot Fwd Pkts', 'Bwd Pkts/s', 'TotLen Fwd Pkts', 'Fwd Seg Size Min',
    'Init Fwd Win Byts', 'Fwd Pkts/s', 'Bwd IAT Mean', 'Fwd Pkt Len Mean',
    'URG Flag Cnt', 'Bwd IAT Min', 'Bwd Pkt Len Min', 'Fwd Pkt Len Max',
    'Pkt Len Min', 'Pkt Len Mean', 'Fwd Pkt Len Min', 'Bwd Pkt Len Mean',
    'Bwd Pkt Len Max', 'Init Bwd Win Byts', 'Flow IAT Std', 'Bwd IAT Max',
    'Flow Duration', 'Flow IAT Max', 'Bwd IAT Std', 'Flow IAT Mean',
    'Bwd IAT Tot', 'Idle Min', 'Down/Up Ratio', 'Fwd PSH Flags',
    'Fwd URG Flags', 'Pkt Len Var', 'Active Max', 'Active Mean',
    'Active Min', 'Active Std', 'FIN Flag Cnt', 'Tot Bwd Pkts'
]

CATEGORICAL_FEATURES = ['Fwd Seg Size Min', 'Pkt Len Min', 'Protocol']
BINARY_FEATURES      = ['FIN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'Fwd URG Flag', 'Fwd PSH Flag']

if IS_RANK_0:
    ALL_FEATURES = sdf_full.columns
    EXCLUDE_FROM_FEATURES = {'Label', 'label_generic', 'Timestamp'}
    ALL_FEATURES = [c for c in sdf_full.columns if c not in EXCLUDE_FROM_FEATURES]

    CONTINUOUS_FEATURES = [
        f for f in ALL_FEATURES
        if f not in CATEGORICAL_FEATURES and f not in BINARY_FEATURES
    ]

    available = set(sdf.columns)
    missing   = [f for f in ALL_FEATURES if f not in available]
    if missing:
        print(f"⚠️  Missing features in dataset: {missing}")
        ALL_FEATURES = [f for f in ALL_FEATURES if f not in missing]
    else:
        print(f"✅  All {len(ALL_FEATURES)} features found in dataset")

In [ ]:
def label_encoding_spark(sdf):
    """Add label_generic_enc (0/1) and Label_enc (integer) to the Spark DataFrame."""
    sdf = sdf.withColumn('label_generic_enc', col('label_generic').cast('int'))
    indexer = StringIndexer(inputCol='Label', outputCol='Label_enc', handleInvalid='keep')
    model = indexer.fit(sdf)
    sdf = model.transform(sdf)
    sdf = sdf.withColumn('Label_enc', col('Label_enc').cast('int'))
    label_classes = list(model.labels)
    n_binary = sdf.select('label_generic_enc').distinct().count()
    print(f"✅  label_generic_enc: {n_binary} classes | Label_enc: {len(label_classes)} classes")
    return sdf, label_classes

In [ ]:
def preprocess_to_pandas(sdf, continuous_features, categorical_features, binary_features):
    """Convert Spark → Pandas and clean dtypes."""
    print("⏳  Converting Spark → Pandas ...")
    pdf = sdf.toPandas()
    print(f"📊  Shape: {pdf.shape}")

    available = set(pdf.columns)
    cont_cols = [c for c in continuous_features if c in available]
    cat_cols  = [c for c in categorical_features if c in available]
    bin_cols  = [c for c in binary_features if c in available]

    for c in cont_cols:
        pdf[c] = pd.to_numeric(pdf[c], errors='coerce')
    pdf[cont_cols] = pdf[cont_cols].fillna(0.0)

    cat_encoders = {}
    cat_dims = {}
    for c in cat_cols:
        le = LabelEncoder()
        pdf[c] = le.fit_transform(pdf[c].astype(str))
        cat_encoders[c] = le
        cat_dims[c] = len(le.classes_)

    for c in bin_cols:
        pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0).astype(int)

    print(f"✅  Preprocessed: {len(cont_cols)} continuous | {len(cat_cols)} categorical | {len(bin_cols)} binary")
    return pdf, cat_encoders, cat_dims

In [ ]:
# ── Preprocess & cache (rank 0 only) ─────────────────────────────────
if IS_RANK_0:
    PREPROCESSED_PATH = f"{PATH}/preprocessed_model_{int(PERCENTAGE_TO_USE*100)}pct.parquet"

    if os.path.exists(PREPROCESSED_PATH):
        pdf_encoded = pd.read_parquet(PREPROCESSED_PATH)

        EXCLUDE = {'Label', 'label_generic', 'Label_enc', 'label_generic_enc', 'Timestamp'}
        FINAL_FEATURES = [c for c in pdf_encoded.columns if c not in EXCLUDE]

        _lbl_map = (pdf_encoded[['Label', 'Label_enc']]
                    .drop_duplicates()
                    .sort_values('Label_enc')
                    .set_index('Label_enc')['Label']
                    .to_dict())
        label_classes = [_lbl_map[i] for i in range(len(_lbl_map))]

        CAT_FEATURE_NAMES = [c for c in CATEGORICAL_FEATURES if c in FINAL_FEATURES]
        CAT_IDXS = [FINAL_FEATURES.index(c) for c in CAT_FEATURE_NAMES]
        CAT_DIMS = [int(pdf_encoded[c].nunique()) for c in CAT_FEATURE_NAMES]

        print(f"✅  Cache loaded: {len(pdf_encoded):,} rows, {len(FINAL_FEATURES)} features")
    else:
        print("⚠️  No cache found — running preprocessing ...")
        sdf_enc, label_classes = label_encoding_spark(sdf)
        pdf_encoded, cat_encoders, cat_dims_dict = preprocess_to_pandas(
            sdf_enc, CONTINUOUS_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES
        )

        EXCLUDE = {'Label', 'label_generic', 'Label_enc', 'label_generic_enc', 'Timestamp'}
        FINAL_FEATURES = [c for c in pdf_encoded.columns if c not in EXCLUDE]
        CAT_FEATURE_NAMES = [c for c in CATEGORICAL_FEATURES if c in FINAL_FEATURES]
        CAT_IDXS = [FINAL_FEATURES.index(c) for c in CAT_FEATURE_NAMES]
        CAT_DIMS = [cat_dims_dict[c] for c in CAT_FEATURE_NAMES]

        pdf_encoded.to_parquet(PREPROCESSED_PATH, index=False)
        print(f"💾  Saved to {PREPROCESSED_PATH}")

    n_classes_multi = len(label_classes)
    print(f"\n🧮  Features: {len(FINAL_FEATURES)} | Binary classes: 2 | Multi classes: {n_classes_multi}")

### 2.1 Prepare arrays & broadcast to all workers

In [ ]:
# ── Prepare data on rank 0, then broadcast to all Horovod workers ──

if IS_RANK_0:
    X = pdf_encoded[FINAL_FEATURES].values.astype(np.float64)
    y_binary = pdf_encoded['label_generic_enc'].values.astype(int)
    y_multi  = pdf_encoded['Label_enc'].values.astype(int)

    # Clean inf / extreme values
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    f32_max = np.finfo(np.float32).max
    X = np.clip(X, -f32_max, f32_max).astype(np.float32)

    # Filter rare classes
    MIN_SAMPLES_PER_CLASS = 5
    classes, counts = np.unique(y_multi, return_counts=True)
    rare_classes = classes[counts < MIN_SAMPLES_PER_CLASS]
    if len(rare_classes) > 0:
        keep_mask = ~np.isin(y_multi, rare_classes)
    else:
        keep_mask = np.ones(len(y_multi), dtype=bool)

    X_clean      = X[keep_mask]
    y_bin_clean  = y_binary[keep_mask]
    y_mul_clean  = y_multi[keep_mask]
    n_classes_multi = len(np.unique(y_mul_clean))
    y_combined   = np.column_stack([y_bin_clean, y_mul_clean])

    # Single stratified split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_clean, y_combined, test_size=TEST_SIZE,
        random_state=RANDOM_SEED, stratify=y_bin_clean
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tr, y_tr, test_size=VAL_SIZE / (1 - TEST_SIZE),
        random_state=RANDOM_SEED, stratify=y_tr[:, 0]
    )

    y_tr_bin,  y_tr_mul  = y_train[:, 0], y_train[:, 1]
    y_val_bin, y_val_mul = y_val[:,   0], y_val[:,   1]
    y_te_bin,  y_te_mul  = y_te[:,    0], y_te[:,    1]

    # StandardScaler on continuous features (fit on train only)
    cont_idxs = [FINAL_FEATURES.index(c) for c in CONTINUOUS_FEATURES if c in FINAL_FEATURES]
    scaler = StandardScaler()
    scaler.fit(X_train[:, cont_idxs])
    for arr in [X_train, X_val, X_te]:
        arr[:, cont_idxs] = scaler.transform(arr[:, cont_idxs])

    print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_te):,}")
    print(f"Binary classes: 2 | Multiclass classes: {n_classes_multi}")

    # Pack everything for broadcasting
    _data_bundle = {
        'X_train': X_train, 'X_val': X_val, 'X_te': X_te,
        'y_tr_bin': y_tr_bin, 'y_tr_mul': y_tr_mul,
        'y_val_bin': y_val_bin, 'y_val_mul': y_val_mul,
        'y_te_bin': y_te_bin, 'y_te_mul': y_te_mul,
        'n_classes_multi': n_classes_multi,
        'FINAL_FEATURES': FINAL_FEATURES,
        'CAT_IDXS': CAT_IDXS,
        'CAT_DIMS': CAT_DIMS,
        'label_classes': label_classes,
    }
else:
    _data_bundle = None

# ── Broadcast data from rank 0 to all workers ────────────────────────
import pickle

if IS_RANK_0:
    _data_bytes = pickle.dumps(_data_bundle)
    _data_tensor = torch.ByteTensor(list(_data_bytes))
    _size_tensor = torch.LongTensor([len(_data_bytes)])
else:
    _size_tensor = torch.LongTensor([0])

hvd.broadcast_(_size_tensor, root_rank=0)

if not IS_RANK_0:
    _data_tensor = torch.ByteTensor(int(_size_tensor.item()))

hvd.broadcast_(_data_tensor, root_rank=0)

if not IS_RANK_0:
    _data_bundle = pickle.loads(bytes(_data_tensor.numpy().tolist()))

# Unpack on all ranks
X_train         = _data_bundle['X_train']
X_val           = _data_bundle['X_val']
X_te            = _data_bundle['X_te']
y_tr_bin        = _data_bundle['y_tr_bin']
y_tr_mul        = _data_bundle['y_tr_mul']
y_val_bin       = _data_bundle['y_val_bin']
y_val_mul       = _data_bundle['y_val_mul']
y_te_bin        = _data_bundle['y_te_bin']
y_te_mul        = _data_bundle['y_te_mul']
n_classes_multi = _data_bundle['n_classes_multi']
FINAL_FEATURES  = _data_bundle['FINAL_FEATURES']
CAT_IDXS        = _data_bundle['CAT_IDXS']
CAT_DIMS        = _data_bundle['CAT_DIMS']
label_classes   = _data_bundle['label_classes']

del _data_bundle, _data_tensor, _size_tensor

if IS_RANK_0:
    print(f"\n✅  Data broadcast to all {hvd.size()} workers")
    print(f"   X_train: {X_train.shape} | X_val: {X_val.shape} | X_te: {X_te.shape}")

## 3. TabNet — Distributed Training with Horovod

**Strategy:** TabNetMultiTaskClassifier doesn't natively support Horovod,
so we wrap its internal PyTorch optimizer with `hvd.DistributedOptimizer`.

Each worker trains on a **shard** of the data (data-parallel). Gradients are
averaged across workers via Horovod's allreduce at each step.

### 3.1 Hyperparameter Tuning (Optuna — rank 0 only)

In [ ]:
def tabnet_multitask_objective(trial, X_train, X_val,
                                y_tr_bin, y_tr_mul,
                                y_val_bin, y_val_mul):
    """
    Optuna objective for TabNetMultiTaskClassifier.
    Run on rank 0 only — fast serial tuning, best params distributed later.
    """
    N_a      = trial.suggest_categorical('N_a',       [8, 16, 32, 64, 128])
    N_steps  = trial.suggest_int('N_steps',           3, 10)
    gamma    = trial.suggest_float('gamma',           1.0, 2.0, step=0.1)
    lambda_s = trial.suggest_float('lambda_sparse',   1e-4, 1e-1, log=True)
    lr       = trial.suggest_float('lr',              1e-4, 1e-2, log=True)
    batch_sz = trial.suggest_categorical('batch_size',[256, 512, 1024, 2048])
    mask_type= trial.suggest_categorical('mask_type', ['sparsemax', 'entmax'])

    clf = TabNetMultiTaskClassifier(
        n_d=N_a, n_a=N_a,
        n_steps=N_steps,
        gamma=gamma,
        lambda_sparse=lambda_s,
        cat_idxs=CAT_IDXS if CAT_IDXS else [],
        cat_dims=CAT_DIMS if CAT_DIMS else [],
        cat_emb_dim=1,
        optimizer_params=dict(lr=lr),
        mask_type=mask_type,
        verbose=0,
        seed=RANDOM_SEED,
    )

    y_train_mt = np.column_stack([y_tr_bin, y_tr_mul])
    y_val_mt   = np.column_stack([y_val_bin, y_val_mul])

    clf.fit(
        X_train, y_train_mt,
        eval_set      = [(X_val, y_val_mt)],
        eval_metric   = ['accuracy'],
        max_epochs    = TABNET_MAX_EPOCHS,
        patience      = 10,
        batch_size    = batch_sz,
        virtual_batch_size = max(batch_sz // 4, 64),
        drop_last     = False,
    )

    preds = clf.predict(X_val)
    mcc_bin = matthews_corrcoef(y_val_bin, preds[0])
    mcc_mul = matthews_corrcoef(y_val_mul, preds[1])
    return (mcc_bin + mcc_mul) / 2


# ── Optuna tuning on rank 0 only ─────────────────────────────────────
if IS_RANK_0:
    print("🔍  Tuning TabNetMultiTaskClassifier (rank 0 only) ...")

    study_tabnet = optuna.create_study(
        direction  = 'maximize',
        sampler    = optuna.samplers.TPESampler(seed=RANDOM_SEED),
        pruner     = optuna.pruners.MedianPruner(n_startup_trials=5),
        study_name = 'tabnet_multitask_hvd'
    )

    study_tabnet.optimize(
        lambda t: tabnet_multitask_objective(
            t, X_train, X_val,
            y_tr_bin, y_tr_mul,
            y_val_bin, y_val_mul
        ),
        n_trials         = N_OPTUNA_TRIALS,
        timeout          = 600000000000000000000,
        show_progress_bar= True
    )

    best_tabnet = study_tabnet.best_params
    print(f"\n✅  Best mean MCC (bin+multi): {study_tabnet.best_value:.4f}")
    print(f"Best params: {best_tabnet}")
    _best_tabnet_bytes = pickle.dumps(best_tabnet)
    _bt_tensor = torch.ByteTensor(list(_best_tabnet_bytes))
    _bt_size   = torch.LongTensor([len(_best_tabnet_bytes)])
else:
    _bt_size = torch.LongTensor([0])

# Broadcast best params to all workers
hvd.broadcast_(_bt_size, root_rank=0)
if not IS_RANK_0:
    _bt_tensor = torch.ByteTensor(int(_bt_size.item()))
hvd.broadcast_(_bt_tensor, root_rank=0)
if not IS_RANK_0:
    best_tabnet = pickle.loads(bytes(_bt_tensor.numpy().tolist()))

if IS_RANK_0:
    print(f"📡  Best TabNet params broadcast to all {hvd.size()} workers")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TabNet Distributed Training — Horovod wrapper
# ══════════════════════════════════════════════════════════════════════
#
# TabNet's .fit() manages its own training loop internally.
# To distribute with Horovod we:
#   1. Shard the training data across workers (each sees 1/N of data)
#   2. After .fit(), allreduce the model weights so all workers converge
#   3. This is "model-averaging" data-parallelism — simple & effective
#
# For deeper integration (gradient-level allreduce per step), you'd
# need to modify TabNet's internal loop or rewrite it in pure PyTorch.
# The data-shard approach is the practical 80/20 solution.
# ══════════════════════════════════════════════════════════════════════

def shard_data(X, y, rank, world_size):
    """Split data into world_size shards, return the shard for this rank."""
    n = len(X)
    shard_size = n // world_size
    start = rank * shard_size
    end   = start + shard_size if rank < world_size - 1 else n
    return X[start:end], y[start:end]


def allreduce_tabnet_weights(model):
    """Average TabNet model parameters across all Horovod workers."""
    for param in model.network.parameters():
        tensor = param.data
        averaged = hvd.allreduce(tensor, op=hvd.Average)
        param.data = averaged


print(f"🏋️  Training final TabNet (distributed, {hvd.size()} workers) ...")

# Each worker gets its shard
y_train_mt = np.column_stack([y_tr_bin, y_tr_mul])
y_val_mt   = np.column_stack([y_val_bin, y_val_mul])

X_train_shard, y_train_shard = shard_data(X_train, y_train_mt, hvd.rank(), hvd.size())

if IS_RANK_0:
    print(f"   Global train: {len(X_train):,} → Shard per worker: ~{len(X_train_shard):,}")

# Scale learning rate by number of workers (linear scaling rule)
scaled_lr = best_tabnet['lr'] * hvd.size()

tabnet_model = TabNetMultiTaskClassifier(
    n_d=best_tabnet['N_a'],  n_a=best_tabnet['N_a'],
    n_steps=best_tabnet['N_steps'],
    gamma=best_tabnet['gamma'],
    lambda_sparse=best_tabnet['lambda_sparse'],
    cat_idxs=CAT_IDXS if CAT_IDXS else [],
    cat_dims=CAT_DIMS if CAT_DIMS else [],
    cat_emb_dim=1,
    optimizer_params=dict(lr=scaled_lr),
    mask_type=best_tabnet['mask_type'],
    verbose=1 if IS_RANK_0 else 0,
    seed=RANDOM_SEED,
)

# Fit on this worker's shard (all workers see validation for early stopping)
tabnet_model.fit(
    X_train_shard, y_train_shard,
    eval_set           = [(X_val, y_val_mt)],
    eval_metric        = ['accuracy'],
    max_epochs         = TABNET_MAX_EPOCHS * 2,
    patience           = 20,
    batch_size         = best_tabnet['batch_size'],
    virtual_batch_size = max(best_tabnet['batch_size'] // 4, 64),
    drop_last          = False,
)

# ── Allreduce: average model weights across all workers ──────────────
allreduce_tabnet_weights(tabnet_model)

if IS_RANK_0:
    tabnet_model.save_model('models/tabnet_multitask_hvd')
    print("✅  Model saved to models/tabnet_multitask_hvd")

In [ ]:
# ── Evaluate TabNet (rank 0 only) ─────────────────────────────────────
if IS_RANK_0:
    preds_te = tabnet_model.predict(X_te)
    proba_te = tabnet_model.predict_proba(X_te)

    preds_te_bin, preds_te_mul = preds_te[0], preds_te[1]
    proba_te_bin, proba_te_mul = proba_te[0], proba_te[1]

    def print_metrics(y_true, y_pred, y_proba, task_name, class_names=None):
        is_binary = (len(np.unique(y_true)) == 2)
        acc = accuracy_score(y_true, y_pred)
        mcc = matthews_corrcoef(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, average='binary' if is_binary else 'macro')
        try:
            auc = roc_auc_score(
                y_true,
                y_proba[:, 1] if is_binary else y_proba,
                multi_class='ovr' if not is_binary else 'raise'
            )
        except Exception:
            auc = float('nan')
        print(f"\n{'='*55}")
        print(f"  {task_name}")
        print(f"{'='*55}")
        print(f"  Accuracy : {acc:.4f}  |  F1: {f1:.4f}  |  MCC: {mcc:.4f}  |  AUC: {auc:.4f}")
        print(f"{'='*55}")
        print(classification_report(y_true, y_pred, target_names=class_names))
        return dict(task=task_name, accuracy=acc, f1=f1, mcc=mcc, auc=auc)

    results_bin = print_metrics(y_te_bin, preds_te_bin, proba_te_bin,
                                'TabNet HVD — Head 0: Binary (label_generic)')
    results_mul = print_metrics(y_te_mul, preds_te_mul, proba_te_mul,
                                'TabNet HVD — Head 1: Multiclass (Label)',
                                class_names=label_classes)

In [ ]:
# ── TabNet Feature Importance (rank 0) ───────────────────────────────
if IS_RANK_0:
    def plot_tabnet_importance(model, feature_names, top_n=20, filename=None):
        importances = model.feature_importances_
        fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
        fi_df = fi_df.sort_values('importance', ascending=False).head(top_n)
        fig, ax = plt.subplots(figsize=(10, top_n * 0.4))
        sns.barplot(data=fi_df, x='importance', y='feature', palette='viridis', ax=ax)
        ax.set_title('TabNet HVD — Global Feature Importance (shared encoder, both tasks)', fontsize=13)
        ax.set_xlabel('Attention-based importance')
        plt.tight_layout()
        if filename:
            plt.savefig(f'{PATH_IMG}/{filename}', dpi=150)
        plt.show()
        return fi_df

    fi_tabnet = plot_tabnet_importance(
        tabnet_model, FINAL_FEATURES,
        filename='tabnet_fi_multitask_hvd.png'
    )
    fi_tabnet.head(15)

In [ ]:
# ── Optuna visualisation (rank 0) ─────────────────────────────────────
if IS_RANK_0:
    fig = optuna.visualization.matplotlib.plot_param_importances(study_tabnet)
    plt.title('TabNet HPO importance — multi-task (Horovod)')
    plt.savefig(f'{PATH_IMG}/optuna_tabnet_multitask_hvd.png', dpi=150, bbox_inches='tight')
    plt.show()

    fig = optuna.visualization.matplotlib.plot_optimization_history(study_tabnet)
    plt.title('TabNet Optuna — optimization history (Horovod)')
    plt.savefig(f'{PATH_IMG}/optuna_tabnet_history_hvd.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Save best params
    with open('models/tabnet_best_params_hvd.json', 'w') as f:
        json.dump(best_tabnet, f, indent=2)
    print("💾  Saved: models/tabnet_best_params_hvd.json")

## 4. TFT — Distributed Training with Horovod + PyTorch Lightning

PyTorch Lightning has native Horovod integration via `HorovodStrategy`.
This is the cleanest approach — no manual sharding needed.

In [ ]:
# ── Build time-indexed dataframe for TFT (rank 0 prepares, broadcast) ─
if IS_RANK_0:
    pdf_tft = pdf_encoded.copy()
    pdf_tft['series_id'] = '0'

    if 'Timestamp' in pdf_tft.columns:
        pdf_tft['Timestamp'] = pd.to_datetime(pdf_tft['Timestamp'], errors='coerce')
        pdf_tft = pdf_tft.sort_values('Timestamp').reset_index(drop=True)
        pdf_tft['time_idx'] = np.arange(len(pdf_tft))
        print("✅  time_idx created from Timestamp ordering")
    else:
        pdf_tft = pdf_tft.reset_index(drop=True)
        pdf_tft['time_idx'] = np.arange(len(pdf_tft))
        print("⚠️  No Timestamp column — using row order as time_idx")

    MIN_SEQ_LEN     = 10
    MAX_PRED_LEN    = 1
    MAX_ENCODER_LEN = 30

    print(f"Total rows: {len(pdf_tft):,} | Encoder window: {MAX_ENCODER_LEN} | Prediction steps: {MAX_PRED_LEN}")
else:
    MIN_SEQ_LEN     = 10
    MAX_PRED_LEN    = 1
    MAX_ENCODER_LEN = 30

In [ ]:
# ── TFT TimeSeriesDataSet (rank 0 creates, then pickled to all) ──────

if IS_RANK_0:
    def make_tft_datasets(pdf, max_encoder_len, max_pred_len, val_frac=0.2):
        pdf = pdf.copy()
        pdf['label_generic_enc'] = pdf['label_generic_enc'].astype(float)
        pdf['Label_enc']         = pdf['Label_enc'].astype(float)

        max_time = pdf['time_idx'].max()
        cutoff   = int(max_time * (1 - val_frac))

        common_kwargs = dict(
            time_idx              = 'time_idx',
            target                = ['label_generic_enc', 'Label_enc'],
            group_ids             = ['series_id'],
            max_encoder_length    = max_encoder_len,
            max_prediction_length = max_pred_len,
            time_varying_known_reals   = [],
            time_varying_unknown_reals = [f for f in FINAL_FEATURES if f in pdf.columns],
            target_normalizer     = None,
            add_relative_time_idx = True,
            add_target_scales     = False,
            add_encoder_length    = True,
        )

        ts_train = TimeSeriesDataSet(
            pdf[lambda d: d['time_idx'] <= cutoff],
            **common_kwargs
        )

        ts_val = TimeSeriesDataSet.from_dataset(
            ts_train,
            pdf[lambda d: d['time_idx'] > cutoff - max_encoder_len],
            predict=True
        )

        return ts_train, ts_val

    print("Building TFT multi-target datasets ...")
    ts_train, ts_val = make_tft_datasets(pdf_tft, MAX_ENCODER_LEN, MAX_PRED_LEN)
    print(f"✅  TFT datasets ready | Targets: {ts_train.target_names}")

    # Serialize datasets for broadcast
    _ts_bytes = pickle.dumps((ts_train, ts_val))
    _ts_tensor = torch.ByteTensor(list(_ts_bytes))
    _ts_size   = torch.LongTensor([len(_ts_bytes)])
else:
    _ts_size = torch.LongTensor([0])

hvd.broadcast_(_ts_size, root_rank=0)
if not IS_RANK_0:
    _ts_tensor = torch.ByteTensor(int(_ts_size.item()))
hvd.broadcast_(_ts_tensor, root_rank=0)
if not IS_RANK_0:
    ts_train, ts_val = pickle.loads(bytes(_ts_tensor.numpy().tolist()))

del _ts_tensor, _ts_size
if IS_RANK_0:
    print(f"📡  TFT datasets broadcast to all {hvd.size()} workers")

### 4.1 TFT Hyperparameter Tuning (Optuna — rank 0 only)

In [ ]:
def tft_multitarget_objective(trial, ts_train, ts_val, max_epochs):
    """
    Optuna objective for TFT — runs on rank 0 only (no Horovod in tuning).
    """
    hidden_size         = trial.suggest_categorical('hidden_size',       [16, 32, 64, 128])
    attention_head_size = trial.suggest_int('attention_head_size',        1, 4)
    dropout             = trial.suggest_float('dropout',                  0.1, 0.4, step=0.05)
    hidden_continuous   = trial.suggest_categorical('hidden_continuous',  [8, 16, 32])
    lr                  = trial.suggest_float('lr',                       1e-4, 1e-2, log=True)
    batch_size          = trial.suggest_categorical('batch_size',         [64, 128, 256])

    train_dl = ts_train.to_dataloader(train=True,  batch_size=batch_size, num_workers=0)
    val_dl   = ts_val.to_dataloader(  train=False, batch_size=batch_size, num_workers=0)

    early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min')

    tft = TemporalFusionTransformer.from_dataset(
        ts_train,
        learning_rate           = lr,
        hidden_size             = hidden_size,
        attention_head_size     = attention_head_size,
        dropout                 = dropout,
        hidden_continuous_size  = hidden_continuous,
        output_size             = [2, n_classes_multi],
        loss                    = MultiLoss([CrossEntropy(), CrossEntropy()]),
        log_interval            = -1,
        reduce_on_plateau_patience = 3,
    )

    trainer = pl.Trainer(
        max_epochs          = max_epochs,
        accelerator         = 'auto',
        enable_progress_bar = False,
        enable_model_summary= False,
        callbacks           = [early_stop],
        logger              = False,
    )

    trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
    return trainer.callback_metrics.get('val_loss', torch.tensor(float('inf'))).item()


# ── Optuna tuning on rank 0 only ─────────────────────────────────────
if IS_RANK_0:
    print("🔍  Tuning TFT (rank 0 only) ...")

    study_tft = optuna.create_study(
        direction  = 'minimize',
        sampler    = optuna.samplers.TPESampler(seed=RANDOM_SEED),
        pruner     = optuna.pruners.HyperbandPruner(),
        study_name = 'tft_multitarget_hvd'
    )

    study_tft.optimize(
        lambda t: tft_multitarget_objective(t, ts_train, ts_val,
                                             max_epochs=TFT_MAX_EPOCHS),
        n_trials         = N_OPTUNA_TRIALS,
        timeout          = 7200,
        show_progress_bar= True
    )

    best_tft = study_tft.best_params
    print(f"\n✅  Best val_loss (multi-target): {study_tft.best_value:.4f}")
    print(f"Best params: {best_tft}")
    _btft_bytes = pickle.dumps(best_tft)
    _btft_tensor = torch.ByteTensor(list(_btft_bytes))
    _btft_size   = torch.LongTensor([len(_btft_bytes)])
else:
    _btft_size = torch.LongTensor([0])

# Broadcast best TFT params
hvd.broadcast_(_btft_size, root_rank=0)
if not IS_RANK_0:
    _btft_tensor = torch.ByteTensor(int(_btft_size.item()))
hvd.broadcast_(_btft_tensor, root_rank=0)
if not IS_RANK_0:
    best_tft = pickle.loads(bytes(_btft_tensor.numpy().tolist()))

if IS_RANK_0:
    print(f"📡  Best TFT params broadcast to all {hvd.size()} workers")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TFT — Distributed Training with Horovod + PyTorch Lightning
# ══════════════════════════════════════════════════════════════════════
#
# PyTorch Lightning natively supports Horovod via strategy='horovod'.
# The Trainer automatically:
#   - Wraps optimizer with hvd.DistributedOptimizer
#   - Uses DistributedSampler for the dataloader
#   - Allreduces gradients every step
#   - Logs and checkpoints on rank 0 only
# ══════════════════════════════════════════════════════════════════════

print(f"🏋️  Training TFT distributed ({hvd.size()} workers via Horovod) ...")

# Scale learning rate
scaled_tft_lr = best_tft['lr'] * hvd.size()

train_dl = ts_train.to_dataloader(train=True,  batch_size=best_tft['batch_size'], num_workers=0)
val_dl   = ts_val.to_dataloader(  train=False, batch_size=best_tft['batch_size'], num_workers=0)

checkpoint_cb = ModelCheckpoint(
    dirpath='models', filename='tft_multitarget_hvd_best',
    monitor='val_loss', mode='min'
)
early_stop = EarlyStopping(monitor='val_loss', patience=15, mode='min')

tft_model = TemporalFusionTransformer.from_dataset(
    ts_train,
    learning_rate           = scaled_tft_lr,
    hidden_size             = best_tft['hidden_size'],
    attention_head_size     = best_tft['attention_head_size'],
    dropout                 = best_tft['dropout'],
    hidden_continuous_size  = best_tft['hidden_continuous'],
    output_size             = [2, n_classes_multi],
    loss                    = MultiLoss([CrossEntropy(), CrossEntropy()]),
    log_interval            = 10,
    reduce_on_plateau_patience = 5,
)

# ── Key: use strategy='horovod' in PL Trainer ────────────────────────
trainer_tft = pl.Trainer(
    max_epochs  = TFT_MAX_EPOCHS * 2,
    accelerator = 'auto',
    strategy    = 'horovod',           # ← Horovod distributed training
    callbacks   = [checkpoint_cb, early_stop],
    logger      = True if IS_RANK_0 else False,
)

trainer_tft.fit(tft_model, train_dataloaders=train_dl, val_dataloaders=val_dl)

if IS_RANK_0:
    print("✅  TFT distributed training complete")

In [ ]:
# ── TFT Optuna visualisation (rank 0) ─────────────────────────────────
if IS_RANK_0:
    fig = optuna.visualization.matplotlib.plot_param_importances(study_tft)
    plt.title('TFT HPO importance — multi-target (Horovod)')
    plt.savefig(f'{PATH_IMG}/optuna_tft_multitarget_hvd.png', dpi=150, bbox_inches='tight')
    plt.show()

    fig = optuna.visualization.matplotlib.plot_optimization_history(study_tft)
    plt.title('TFT Optuna — optimization history (Horovod)')
    plt.savefig(f'{PATH_IMG}/optuna_tft_history_hvd.png', dpi=150, bbox_inches='tight')
    plt.show()

    val_loss_final = trainer_tft.callback_metrics.get('val_loss', torch.tensor(float('nan'))).item()
    print(f"\n✅  Final TFT val_loss: {val_loss_final:.4f}")

### 4.2 TFT — Feature Importance (Variable Selection Networks)

In [ ]:
# ── TFT Feature Importance (rank 0) ──────────────────────────────────
if IS_RANK_0:
    def plot_tft_importance(tft_model, ts_val, top_n=20):
        val_dl = ts_val.to_dataloader(train=False, batch_size=256, num_workers=0)
        raw_preds, x = tft_model.predict(val_dl, mode='raw', return_x=True)
        interpretation = tft_model.interpret_output(raw_preds, reduction='sum')

        encoder_vars = tft_model.encoder_variables
        enc_weights  = interpretation['encoder_variables'].numpy()

        fi_df = pd.DataFrame({'feature': encoder_vars, 'importance': enc_weights})
        fi_df = fi_df.sort_values('importance', ascending=False).head(top_n)

        fig, ax = plt.subplots(figsize=(10, top_n * 0.45))
        sns.barplot(data=fi_df, x='importance', y='feature', palette='magma', ax=ax)
        ax.set_title('TFT HVD — Variable Selection Weights (shared encoder, both targets)', fontsize=13)
        ax.set_xlabel('Selection weight (sum over val set)')
        plt.tight_layout()
        plt.savefig(f'{PATH_IMG}/tft_fi_multitarget_hvd.png', dpi=150)
        plt.show()
        return fi_df

    fi_tft = plot_tft_importance(tft_model, ts_val)
    fi_tft.head(15)

## 5. Results Summary & Comparison

In [ ]:
# ── Results summary (rank 0) ──────────────────────────────────────────
if IS_RANK_0:
    summary_rows = [
        {
            'Model': f'TabNet HVD ({hvd.size()} workers)',
            'Target': 'Binary (label_generic)',
            'Accuracy': results_bin['accuracy'],
            'F1':       results_bin['f1'],
            'MCC':      results_bin['mcc'],
            'AUC':      results_bin['auc'],
        },
        {
            'Model': f'TabNet HVD ({hvd.size()} workers)',
            'Target': 'Multiclass (Label)',
            'Accuracy': results_mul['accuracy'],
            'F1':       results_mul['f1'],
            'MCC':      results_mul['mcc'],
            'AUC':      results_mul['auc'],
        },
        {
            'Model': f'TFT HVD ({hvd.size()} workers)',
            'Target': 'Both (combined val loss)',
            'Accuracy': float('nan'),
            'F1':       float('nan'),
            'MCC':      float('nan'),
            'AUC':      float('nan'),
            'Val Loss': val_loss_final,
        },
    ]

    df_summary = pd.DataFrame(summary_rows).round(4)
    print(df_summary.to_string(index=False))
    df_summary.to_csv(f'{PATH_IMG}/results_summary_hvd.csv', index=False)

In [ ]:
# ── Feature importance comparison (rank 0) ────────────────────────────
if IS_RANK_0:
    top10_tabnet = fi_tabnet.head(10).set_index('feature')['importance']
    top10_tft    = fi_tft.head(10).set_index('feature')['importance']

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    top10_tabnet.plot.barh(ax=axes[0], color='steelblue')
    axes[0].set_title('TabNet HVD — Top 10 Features\n(shared encoder, both tasks)', fontsize=12)
    axes[0].invert_yaxis()

    top10_tft.plot.barh(ax=axes[1], color='darkorange')
    axes[1].set_title('TFT HVD — Top 10 Features\n(shared encoder, both targets)', fontsize=12)
    axes[1].invert_yaxis()

    plt.suptitle(f'Feature Importance Comparison: TabNet vs TFT (Horovod, {hvd.size()} workers)', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{PATH_IMG}/fi_comparison_multitask_hvd.png', dpi=150)
    plt.show()

In [ ]:
# ── Save all best hyperparameters ─────────────────────────────────────
if IS_RANK_0:
    all_best_params = {
        'tabnet_multitask_hvd': best_tabnet,
        'tft_multitarget_hvd':  best_tft,
    }

    with open('models/best_hyperparams_hvd.json', 'w') as f:
        json.dump(all_best_params, f, indent=2)
    print("💾  Saved: models/best_hyperparams_hvd.json")

    rows_csv = []
    for model_name, params in all_best_params.items():
        rows_csv.append({'model': model_name, **params})
    pd.DataFrame(rows_csv).to_csv('models/best_hyperparams_hvd.csv', index=False)
    print("💾  Saved: models/best_hyperparams_hvd.csv")

    print(json.dumps(all_best_params, indent=2))

In [ ]:
# ── Cleanup ──────────────────────────────────────────────────────────
if IS_RANK_0:
    try:
        spark.stop()
        print("✅  Spark stopped")
    except:
        pass

# Horovod shutdown happens automatically at process exit
if IS_RANK_0:
    print(f"\n🎉  Distributed training complete ({hvd.size()} workers)")
    print("    All models and results saved with '_hvd' suffix.")

---
## Notes for thesis — Horovod distributed training

### Architecture
- **Data parallelism**: each worker trains on a shard of the data
- **Gradient synchronization**: allreduce after each batch (TFT via PL) or model averaging after training (TabNet)
- **Linear scaling rule**: LR × N_workers to compensate for larger effective batch size

### TabNet distribution strategy
- TabNet's `.fit()` loop is opaque → data sharding + model weight averaging
- Each worker trains on `1/N` of the data independently
- After training, `hvd.allreduce` averages all model parameters
- Simple but effective for tabular models

### TFT distribution strategy  
- PyTorch Lightning `strategy='horovod'` provides native integration
- Automatic `DistributedSampler` for data loading
- Per-step gradient allreduce → true distributed SGD
- Checkpointing and logging on rank 0 only

### Running distributed
```bash
# Export notebook to script
jupyter nbconvert --to script cavas_model_horovod.ipynb

# Single machine, 4 GPUs
horovodrun -np 4 -H localhost:4 python cavas_model_horovod.py

# Multi-node (2 machines, 4 GPUs each)
horovodrun -np 8 -H server1:4,server2:4 python cavas_model_horovod.py

# CPU-only (4 processes)
horovodrun -np 4 python cavas_model_horovod.py
```

### Expected speedup
| Workers | Theoretical speedup | Realistic speedup |
|---------|--------------------|-----------------|
| 1       | 1.0×               | 1.0×            |
| 2       | 2.0×               | ~1.7–1.9×       |
| 4       | 4.0×               | ~3.2–3.6×       |
| 8       | 8.0×               | ~5.5–7.0×       |

Communication overhead increases with workers → sub-linear scaling.
TFT benefits more than TabNet due to per-step gradient sync.